In [2]:
import requests
from bs4 import BeautifulSoup
import csv
from pathlib import Path

In [3]:
SOURCE_URL = "https://realpython.github.io/fake-jobs/"

print(SOURCE_URL)

https://realpython.github.io/fake-jobs/


In [4]:
response = requests.get(SOURCE_URL, timeout=10)

response.raise_for_status()

print("Page downloaded successfully.")
print("Status code:", response.status_code)

Page downloaded successfully.
Status code: 200


In [5]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.get_text(strip=True))

Fake Python


In [6]:
job_cards = soup.find_all("div", class_="card-content")

print("Job cards found:", len(job_cards))

Job cards found: 100


In [7]:
print(job_cards[0].prettify()[:3000])

<div class="card-content">
 <div class="media">
  <div class="media-left">
   <figure class="image is-48x48">
    <img alt="Real Python Logo" src="https://files.realpython.com/media/real-python-logo-thumbnail.7f0db70c2ed2.jpg?__no_cf_polish=1"/>
   </figure>
  </div>
  <div class="media-content">
   <h2 class="title is-5">
    Senior Python Developer
   </h2>
   <h3 class="subtitle is-6 company">
    Payne, Roberts and Davis
   </h3>
  </div>
 </div>
 <div class="content">
  <p class="location">
   Stewartbury, AA
  </p>
  <p class="is-small has-text-grey">
   <time datetime="2021-04-08">
    2021-04-08
   </time>
  </p>
 </div>
 <footer class="card-footer">
  <a class="card-footer-item" href="https://www.realpython.com" target="_blank">
   Learn
  </a>
  <a class="card-footer-item" href="https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html" target="_blank">
   Apply
  </a>
 </footer>
</div>



In [8]:
job = job_cards[0]

title = job.find("h2", class_="title").get_text(strip=True)
company = job.find("h3", class_="company").get_text(strip=True)
location = job.find("p", class_="location").get_text(strip=True)
job_url = job.find("a").get("href")

print("Job Title:", title)
print("Company:", company)
print("Location:", location)
print("Job URL:", job_url)

Job Title: Senior Python Developer
Company: Payne, Roberts and Davis
Location: Stewartbury, AA
Job URL: https://www.realpython.com


In [9]:
job_listings = []

for job in job_cards:
    title_element = job.find("h2", class_="title")
    company_element = job.find("h3", class_="company")
    location_element = job.find("p", class_="location")
    link_element = job.find("a")

    job_listings.append({
        "job_title": title_element.get_text(strip=True) if title_element else None,
        "company": company_element.get_text(strip=True) if company_element else None,
        "location": location_element.get_text(strip=True) if location_element else None,
        "job_url": link_element.get("href") if link_element else None
    })

print("Jobs scraped:", len(job_listings))

Jobs scraped: 100


In [10]:
job_listings[:3]

[{'job_title': 'Senior Python Developer',
  'company': 'Payne, Roberts and Davis',
  'location': 'Stewartbury, AA',
  'job_url': 'https://www.realpython.com'},
 {'job_title': 'Energy engineer',
  'company': 'Vasquez-Davidson',
  'location': 'Christopherville, AA',
  'job_url': 'https://www.realpython.com'},
 {'job_title': 'Legal executive',
  'company': 'Jackson, Chambers and Levy',
  'location': 'Port Ericaburgh, AA',
  'job_url': 'https://www.realpython.com'}]

In [11]:
for field in ["job_title", "company", "location", "job_url"]:
    missing = sum(
        1 for job in job_listings
        if not job[field]
    )

    print(f"{field}: {missing} missing")

job_title: 0 missing
company: 0 missing
location: 0 missing
job_url: 0 missing


In [12]:
data_folder = Path("data")
data_folder.mkdir(exist_ok=True)


In [13]:
csv_file = data_folder / "job_listings.csv"

fieldnames = [
    "job_title",
    "company",
    "location",
    "job_url"
]

with open(csv_file, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(job_listings)

print(f"CSV saved to: {csv_file}")

CSV saved to: data\job_listings.csv


In [14]:
with open(csv_file, "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    saved_jobs = list(reader)

print("Records saved:", len(saved_jobs))
print("\nFirst record:")
print(saved_jobs[0])

Records saved: 100

First record:
{'job_title': 'Senior Python Developer', 'company': 'Payne, Roberts and Davis', 'location': 'Stewartbury, AA', 'job_url': 'https://www.realpython.com'}
